# Week 10 Demo: A workflow can run and still be wrong
Every cell in this notebook runs without an error. That does not mean every decision in it is correct.
We will test four assumptions: that dropping incomplete rows removes only bad records, that pandas shows us what the file contains,
that a validator which rejects nothing is working, and that a high score means a good model.
The data is a separate 6,000-row SecurePayATL extract. Read DATA_GUIDE.txt.
Pause at each question before running the next cell. All outputs should remain visible.

In [ ]:
from io import StringIO
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
print('scikit-learn:', sklearn.__version__)

In [ ]:
raw = pd.read_csv('data/Week_10_Demo_Transactions.csv')
print(raw.shape)
print(raw['is_fraud'].value_counts())

## 1. Predict, then inspect
A colleague requires every location field to be present. Will that remove only invalid transactions?
We are auditing a suggestion, so we keep the original raw table.

In [ ]:
location_fields = ['ip_city', 'ip_state', 'ip_latitude', 'ip_longitude']
proposed = raw.dropna(subset=location_fields)
lost = raw.loc[~raw.transaction_id.isin(proposed.transaction_id)]
print('Before:', len(raw), 'After:', len(proposed), 'Lost:', len(lost))
print('Fraud labels before:', raw.is_fraud.sum(), 'after:', proposed.is_fraud.sum(), 'lost:', lost.is_fraud.sum())
print(lost[['transaction_id', 'amount', 'ip_city', 'ip_state']].head().to_string(index=False))
print(lost[['ip_city', 'ip_state']].fillna('<missing>').value_counts())

## 2. Interpret the evidence
The rule removed valid transactions, including some of our rare fraud cases. A positive amount does not become invalid because its location is missing.
For this exercise, missing location information is allowed. We keep those transactions as long as their required numeric values are valid.

Look at what the lost rows contain. XX with International is a documented value: the IP address was outside the US. Rows with XX lost their coordinates, not their state.
UNKNOWN, 0.0.0.0, and VPN in ip_city are placeholders, but pandas reads them as ordinary text. They are not missing to pandas.

## 3. What does pandas call missing?
The reverse also happens: pandas converts some text to missing values while reading the file.
Predict which of these four status values will appear as missing.

In [ ]:
sample_csv = 'id,status\n1,None\n2,NA\n3,Filed\n4,\n'
print(pd.read_csv(StringIO(sample_csv)))
print(pd.read_csv(StringIO(sample_csv), keep_default_na=False))

By default, pandas treats text such as None, NA, null, and an empty cell as missing. With keep_default_na=False, it keeps the text that is in the file.
When a column is mostly missing, open the CSV as text and check whether the values are absent or whether loading changed them. Then check DATA_GUIDE.txt for what the text means.

## 4. Validate the required values
This validation uses fixed rules and does not learn from test data.
The categorical imputer and scaler will learn or store their preparation settings inside the fitted pipeline.

In [ ]:
NUMERIC = ['amount', 'account_age_days', 'transaction_count_24hr']
CATEGORICAL = ['merchant_category', 'ip_state']
FEATURES = NUMERIC + CATEGORICAL
OUTCOME_FIELDS = {'is_fraud', 'fraud_confirm_method', 'chargeback_status'}

def validate_records(frame):
    missing = [c for c in ['transaction_id'] + FEATURES if c not in frame.columns]
    if missing:
        raise ValueError(f'Missing required columns: {missing}')
    clean = frame.copy()
    reasons = pd.Series('', index=clean.index, dtype='object')
    for col in NUMERIC:
        values = pd.to_numeric(clean[col], errors='coerce')
        invalid = values.isna() | ~np.isfinite(values)
        if col == 'amount':
            invalid |= values.le(0)
        else:
            invalid |= values.lt(0) | values.mod(1).ne(0)
        reasons.loc[invalid] += col + ': invalid required value; '
        clean[col] = values
    clean['merchant_category'] = clean['merchant_category'].replace(
        ['UNKNOWN', 'MCC_9999', 'UNDEFINED'], np.nan)
    rejected = clean.loc[reasons.ne(''), ['transaction_id']].copy()
    rejected['reason'] = reasons.loc[reasons.ne('')].str.rstrip('; ')
    valid = clean.loc[reasons.eq('')].copy()
    return valid, rejected

def make_model(numeric=NUMERIC, categorical_features=CATEGORICAL):
    categorical = Pipeline([
        ('impute', SimpleImputer(strategy='constant', fill_value='Missing')),
        ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    preparation = ColumnTransformer([
        ('numeric', StandardScaler(), numeric),
        ('categorical', categorical, categorical_features)
    ], verbose_feature_names_out=False)
    return Pipeline([
        ('prepare', preparation),
        ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))
    ])

def metric_row(name, actual, predicted):
    return {'workflow': name, 'accuracy': accuracy_score(actual, predicted),
            'fraud_precision': precision_score(actual, predicted, zero_division=0),
            'fraud_recall': recall_score(actual, predicted, zero_division=0)}

In [ ]:
valid, rejected = validate_records(raw)
print('Incoming:', len(raw), 'Valid:', len(valid), 'Rejected:', len(rejected))

Zero rejections. Is that good news? It could mean the data is clean, or it could mean the validator never rejects anything.
The output alone cannot tell us which. (DATA_GUIDE.txt explains that this extract contains no invalid numeric records.)
Before we trust the validator, we give it records where we already know the right answer. Predict which IDs it should reject.

In [ ]:
known_cases = pd.DataFrame({
    'transaction_id': ['T_NEGATIVE', 'T_ZERO', 'T_BLANK', 'T_HALF_COUNT', 'T_INFINITE', 'T_GOOD'],
    'amount': [-5, 0, np.nan, 10, np.inf, 10],
    'account_age_days': [30, 30, 30, 30, 30, 30],
    'transaction_count_24hr': [1, 1, 1, 1.5, 1, 1],
    'merchant_category': ['Grocery'] * 6,
    'ip_state': ['GA'] * 6})
known_valid, known_rejected = validate_records(known_cases)
print(known_rejected.to_string(index=False))
expected_rejections = {'T_NEGATIVE', 'T_ZERO', 'T_BLANK', 'T_HALF_COUNT', 'T_INFINITE'}
print('Rejected exactly the expected IDs:', set(known_rejected.transaction_id) == expected_rejections)

## 5. Train and compare
The model and its settings are provided. Class weighting gives rare fraud cases more training emphasis.
We keep the original test prevalence. A high baseline accuracy can coexist with zero fraud recall.
There is no required metric threshold in this lesson.

In [ ]:
X = valid[FEATURES]
y = valid['is_fraud']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42)
model = make_model()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)
metrics = pd.DataFrame([metric_row('Logistic regression', y_test, predictions),
                        metric_row('Most frequent baseline', y_test, baseline_predictions)])
print(metrics.to_string(index=False))
print('Test fraud cases:', int(y_test.sum()))
print('Rows = actual [False, True]; columns = predicted [False, True]')
print(confusion_matrix(y_test, predictions, labels=[False, True]))

The test set has very few fraud cases, so each one changes recall by a large step. Read the confusion matrix counts, not only the percentages.

## 6. A result that looks too good
A colleague adds chargeback_status as a feature and reports a much better model. Predict the scores before running the cell.
We use the same valid rows and the same split settings, so both models are tested on the same transactions.

In [ ]:
LEAKY_CATEGORICAL = CATEGORICAL + ['chargeback_status']
X_leaky = valid[NUMERIC + LEAKY_CATEGORICAL]
X_leaky_train, X_leaky_test, _, _ = train_test_split(
    X_leaky, y, test_size=0.20, stratify=y, random_state=42)
print('Same test rows:', X_leaky_test.index.equals(X_test.index))
leaky_model = make_model(categorical_features=LEAKY_CATEGORICAL)
leaky_model.fit(X_leaky_train, y_train)
leaky_predictions = leaky_model.predict(X_leaky_test)
print(pd.DataFrame([metric_row('Adds chargeback_status', y_test, leaky_predictions),
                    metric_row('Five approved features', y_test, predictions)]).to_string(index=False))
print(confusion_matrix(y_test, leaky_predictions, labels=[False, True]))
print(pd.crosstab(raw.chargeback_status.fillna('<missing>'), raw.is_fraud))

The code ran, the split was stratified, and the scores are excellent. The model is still unusable for our task.
A chargeback is filed after a transaction, often after an investigation. When a transaction arrives, this value does not exist yet.
The cross-tabulation shows why the score is so high: the field nearly records the answer. A near-perfect score on a rare, hard outcome is a reason to investigate, not to celebrate.
We keep the five-feature model, even though its scores are lower. Its scores are an honest estimate for the task we actually have.

## 7. Verify the repair
A useful check must be able to fail. We run the same checks on the proposed workflows and on the repaired one.
The record check asks whether every raw ID was either kept or logged with a reason, exactly once. The feature check reads the columns the fitted model actually received, not a list we typed.

In [ ]:
def every_id_accounted_for(kept, logged, source):
    ids = pd.concat([kept.transaction_id, logged.transaction_id])
    return ids.is_unique and set(ids) == set(source.transaction_id)

def inputs_exclude_outcomes(fitted_model):
    return set(fitted_model.named_steps['prepare'].feature_names_in_).isdisjoint(OUTCOME_FIELDS)

no_log = raw.loc[[], ['transaction_id']]
checks = pd.DataFrame({
    'check': ['Every raw ID kept or logged once', 'Model inputs exclude outcome fields'],
    'proposed': [every_id_accounted_for(proposed, no_log, raw), inputs_exclude_outcomes(leaky_model)],
    'repaired': [every_id_accounted_for(valid, rejected, raw), inputs_exclude_outcomes(model)]})
print(checks.to_string(index=False))

In [ ]:
assert checks['repaired'].all()
assert set(X_train.index).isdisjoint(X_test.index)
prepared = model.named_steps['prepare'].transform(X_test)
assert prepared.shape[0] == len(X_test)
assert np.isfinite(prepared).all()
assert len(predictions) == len(X_test)
print('Verification passed. These checks do not establish production readiness.')

The split check confirms separate row indices. It does not demonstrate that repeated users are independent.

## 8. A small incoming batch
Predict first: which records should receive predictions? What happens to a new merchant category?
The new category is not a reason to refit the encoder.

In [ ]:
incoming = pd.DataFrame([
 {'transaction_id':'DEMO_A','amount':25,'account_age_days':60,'transaction_count_24hr':2,'merchant_category':'Grocery','ip_state':np.nan},
 {'transaction_id':'DEMO_B','amount':70,'account_age_days':20,'transaction_count_24hr':1,'merchant_category':'New_Category','ip_state':'GA'},
 {'transaction_id':'DEMO_C','amount':-8,'account_age_days':20,'transaction_count_24hr':1,'merchant_category':'Grocery','ip_state':'GA'}
])
new_valid, new_rejected = validate_records(incoming)
before = model.named_steps['prepare'].get_feature_names_out().copy()
output = new_valid[['transaction_id']].copy()
output['predicted_fraud'] = model.predict(new_valid[FEATURES])
after = model.named_steps['prepare'].get_feature_names_out()
assert np.array_equal(before, after)
assert every_id_accounted_for(output, new_rejected, incoming)
print(output.to_string(index=False))
print(new_rejected.to_string(index=False))

The feature-name check passed because we reused the fitted preparation. Would it catch a mistake?
Here is the suggested shortcut: fit a new encoder on the incoming batch so that New_Category gets its own column.

In [ ]:
batch_encoder = make_model().named_steps['prepare'].fit(new_valid[FEATURES])
batch_names = batch_encoder.get_feature_names_out()
print('Fitted pipeline columns:', len(before), ' Batch encoder columns:', len(batch_names))
print('Feature names unchanged:', np.array_equal(before, batch_names))
print('Batch encoder columns:', list(batch_names))

The check returns False, so it detects the mistake. The batch encoder learned different columns and scaled the numbers using only two records.
The trained model's coefficients belong to the original columns, so these inputs no longer mean what the model expects.

## Discussion answers
1. **Did removing incomplete locations solve a validity problem?** No. It removed valid transactions, including fraud cases. Use the lost-row and lost-fraud counts.
2. **Does a missing value in pandas mean the file was empty?** Not always. pandas converts text such as None and NA to missing. Check the file and the data guide.
3. **Does zero rejections prove the validator works?** No. Testing it with records whose correct outcome we already know does.
4. **Why split before fitting?** The scaler and encoder must learn from training rows. Test rows remain reserved for evaluation.
5. **Why reject the model with the better score?** It uses chargeback_status, which is recorded after the transaction. Its score measures leakage, not the task.
6. **Does a new category require a new encoder?** No. The fitted encoder with handle_unknown='ignore' keeps the same columns and uses zeros for that feature's unseen category.
7. **What does a low fraud precision mean?** Many flagged transactions were legitimate. Read the actual printed precision and confusion matrix before making a claim.
8. **Which incoming record is rejected?** DEMO_C violates amount > 0. DEMO_A and DEMO_B receive predictions.